# 8 · TF-IDF + cosine

**Owner:** Member 2 · **Branch:** `dev-tfidf` · **Code:** `src/discovery/rank/tfidf.py` → class `TfidfRanker` · **Writes to:** `eval/results/tfidf.json`

**What it means:** Each repository becomes a vector with one number per word: how often the word appears in it (TF), times how rare the word is across all repositories (IDF). The query becomes a vector the same way, and repositories are ranked by the cosine of the angle between the two vectors.

**Key idea:** Rare words count more: "nestjs" says more about a repository than "app". Cosine compares direction, not length, so a long README doesn't win just by being long. It is the classic baseline BM25 should beat (hypothesis H1).

Full guide with starter code, and the Results sheet: [team plan](https://claude.ai/artifact/Rst8GHxAbeHavg4JS2RZV3) (tab 8 TF-IDF + cosine). Handoff and setup: `docs/handoff-methods.md`.

### Settings to try
| Setting | Values | Why |
|---|---|---|
| tf | raw count vs 1 + log(count) | Sublinear tf stops a word repeated 50 times from counting 50×. |
| idf | log((1 + N) / (1 + df)) + 1 | Smooth idf, as in scikit-learn. |
| field weights | README only vs name + topics repeated 3× | Words in the name and topics are stronger evidence than a word deep in the README. |
| min_df | 1, 2, 5 | Drop words that appear in fewer repositories (typos, ids). |

**Fixed:** our own implementation (numpy + scipy.sparse), checked against scikit-learn's TfidfVectorizer

### What we agreed (same for all five)
- **One tokenizer:** `tokenize()` from `discovery.preprocess.text`, for documents *and* queries. Never a private one.
- **One interface:** your class extends `SearchMethod` (`src/discovery/rank/base.py`): `fit(corpus)` → self, `search(query, k)` → `[{"doc_id": full_name, "score": float}]`, best first. `check_hits()` checks it.
- **Tune on the 5 development queries only** (`DEV`, replaced by the real ones in Phase 13). The 20 test queries are for the final number only.
- **Seed 42**, paths from `PATHS`, and don't edit the shared files (`rank/base.py`, `rank/devkit.py`, `preprocess/text.py`, `config/config.yaml`) on your branch: ask Member 6.
- Commit this notebook **with its outputs** on `dev-tfidf`. No data files in git.

## 1. Setup: load the corpus and the shared tokenizer

**Check:** it prints 8,223 repositories, and `node.js` and `c#` come out of `tokenize()` in one piece.

In [ ]:
%load_ext autoreload
%autoreload 2

import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from discovery.config import PATHS, SEED, set_seed
from discovery.preprocess.text import doc_text, tokenize      # the ONE shared tokenizer (Phase 6)
from discovery.rank.base import check_hits
from discovery.rank.devkit import DEV, load_corpus, measure, save_results, show_top
from discovery.rank.tfidf import TfidfRanker

set_seed()
corpus = load_corpus()                                   # data/processed/corpus.parquet
ids = corpus["full_name"].tolist()                       # doc_id = full_name, "owner/name"
text = doc_text(corpus).tolist()                         # name + description + topics + README
print(f"{len(corpus):,} repositories |", tokenize("Built with Node.js and C#"))
print("development queries:", *DEV, sep="\n  ")

## 2. Build: TF-IDF + cosine

One sparse row per repository. After scaling every row to length 1, the cosine between two vectors is just their dot product.

**Check:** it runs once on the whole corpus without errors. Write down how long it took: it goes in the sheet.

<details><summary>Starter code from the team plan (open it if you are stuck; adapt it, don't just paste)</summary>

```python
import numpy as np
from collections import Counter
from scipy import sparse

docs = [tokenize(t) for t in text]
vocab = {w: j for j, w in enumerate(sorted({w for d in docs for w in d}))}
N = len(docs)
rows, cols, vals = [], [], []
for i, d in enumerate(docs):
    for w, c in Counter(d).items():
        rows.append(i); cols.append(vocab[w]); vals.append(1 + np.log(c))     # sublinear tf
tf = sparse.csr_matrix((vals, (rows, cols)), shape=(N, len(vocab)))
df = np.bincount(tf.indices, minlength=len(vocab))                             # repositories per word
idf = np.log((1 + N) / (1 + df)) + 1
X = tf.multiply(idf).tocsr()
X = sparse.diags(1 / np.sqrt(X.multiply(X).sum(1).A1 + 1e-12)) @ X               # unit length: cosine = dot
print(X.shape, f"{X.nnz / N:.0f} non-zero words per repository")
```
</details>

## 3. Search: a free-text query in, a ranked list out

The query gets the same weights. One sparse matrix–vector product scores all repositories; keep the top k with a score above 0.

**Check:** the top 10 for `DEV[0]` look plausible when you read their names and descriptions.

<details><summary>Starter code from the team plan (open it if you are stuck; adapt it, don't just paste)</summary>

```python
def search(query, k=10):
    q = np.zeros(len(vocab))
    for w, c in Counter(tokenize(query)).items():
        if w in vocab:
            q[vocab[w]] = (1 + np.log(c)) * idf[vocab[w]]
    if not q.any():
        return []
    s = X @ (q / np.linalg.norm(q))                       # cosine with every repository at once
    top = np.argsort(-s)[:k]
    return [{"doc_id": ids[i], "score": float(s[i])} for i in top if s[i] > 0]

search(DEV[0])
```
</details>

## 4. Move it into `src/discovery/rank/tfidf.py` as `TfidfRanker`

The stub is already there with the right signature. Fill in `fit` and `search` (and `params`), then load it here with `discovery.rank.tfidf.TfidfRanker`: autoreload picks up your edits.

**Check:** `check_hits` passes below, and in a terminal `pytest tests/test_methods.py -k tfidf` passes (it was skipped until now).

<details><summary>The search method, as in the plan (open it if you are stuck; adapt it, don't just paste)</summary>

```python
    def search(self, query, k=10):
        ...  # the function above, using self.vocab, self.idf, self.X, self.ids
```
</details>

In [ ]:
method = TfidfRanker().fit(corpus)
check_hits(method.search(DEV[0], 10), k=10)
method.params()

## 5. Compare with a reference: scikit-learn

With the same tokenizer and settings, `TfidfVectorizer` should give almost the same ranking. Small differences can come from ties.

**Check:** 9 or 10 of 10 the same on every development query. If not, compare idf first: print both for one word.

<details><summary>The check, as in the plan (open it if you are stuck; adapt it, don't just paste)</summary>

```python
from sklearn.feature_extraction.text import TfidfVectorizer

ref = TfidfVectorizer(analyzer=tokenize, sublinear_tf=True).fit(text)
R = ref.transform(text)
for q in DEV:
    ours = [h["doc_id"] for h in search(q)]
    s = (R @ ref.transform([q]).T).toarray().ravel()
    theirs = [ids[i] for i in np.argsort(-s)[:10]]
    print(f"{len(set(ours) & set(theirs)):2d}/10 the same | {q}")
```
</details>

## 6. Read your top 10 on the development queries

Read them. Write down two good results and two bad ones, and why each bad one ranked high (in the markdown cell under the output).

**Check:** you can say in one sentence why each bad result ranked high.

In [ ]:
show_top(method, corpus, DEV)

*Two good results, two bad results, and why:*

- 
- 
- 
- 

## 7. Tune on the 5 development queries only

Try each setting in the table on the 5 development queries and keep what raises mean NDCG@10. Field weights usually matter most.

Until Phase 13 gives the development labels, compare settings by reading the top 10 (step 6). When `eval/qrels.csv` has the development labels, use `discovery.evaluation.metrics.ndcg_at_k` and keep the setting with the best mean NDCG@10. **Never tune on the 20 test queries.**

**Check:** a small table: each setting tried, what changed in the top 10 (later: mean NDCG@10). Keep the best and note it in the sheet.

## 8. Speed, results file, hand in

`measure()` times the build and the queries; `save_results()` writes `eval/results/tfidf.json` in the format the Results sheet imports. The metrics stay empty until Phase 13–14: then Member 6's `evaluate_method(method, split="test")` fills them in and you import the file in the **Results sheet** tab of the team plan, add your observations and download the PDF.

**Check:** `eval/results/tfidf.json` exists; this notebook is committed with its outputs on `dev-tfidf`; `pytest` passes.

In [ ]:
speed = measure(lambda: TfidfRanker().fit(corpus), DEV)
print(speed)
save_results(method, speed)

## Extra (optional): Why cosine and not dot product?

Rank once without the length scaling (plain dot product). Which repositories rise to the top, and how long are their READMEs? It is a good paragraph for the report on document length.

## What we found

*(fill in at the end: build time, the best setting, two good and two bad results, and anything surprising)*